# M2 — Category  (summary-only, group-aware split — honest score)

Based on the **Category** notebook: summary-only TF-IDF + logistic regression
with a **GroupShuffleSplit on the summary**. This reports the *honest* category
accuracy (~0.80 macro-F1), without the `tags` leakage that inflates a naive
model to ~0.95. Saves `models/m2_category.joblib`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import northstar_prep as P
from sklearn.metrics import f1_score, accuracy_score

df = P.load_clean()
FEATS, TARGET = ["case_summary"], "category"
tr, te = P.group_split(df)
pipe = P.make_clf([P.t_word(), P.t_char()]).fit(tr[FEATS], tr[TARGET])
pred = pipe.predict(te[FEATS])
metrics = {"macro_F1": round(f1_score(te[TARGET], pred, average="macro"), 3),
           "accuracy": round(accuracy_score(te[TARGET], pred), 3),
           "weighted_F1": round(f1_score(te[TARGET], pred, average="weighted"), 3),
           "baseline_acc": round(df[TARGET].value_counts(normalize=True).iloc[0], 3)}
print(metrics)

In [ ]:
proba = pipe.predict_proba(te[FEATS])
test = te[["case_id","case_summary",TARGET]].copy().reset_index(drop=True)
test["predicted"]  = pred
test["confidence"] = proba.max(axis=1).round(3)

full = P.make_clf([P.t_word(), P.t_char()]).fit(df[FEATS], df[TARGET])
path = P.save_bundle("m2_category", {
    "model": full, "target": TARGET, "features": FEATS, "classes": sorted(df[TARGET].unique()),
    "kind": "multiclass", "label": "Category", "metrics": metrics, "test": test,
    "meta": {"author":"Category-notebook","split":"group-on-summary","inputs":"summary only (no tags leakage)"}})
print("saved", path, "| test rows:", len(test))